# Notebook 2A — Tuning K-Nearest Neighbors

*Day 4 · Student A's lab — read the docstring, choose the knobs, implement the search, and tune KNN deeply.*

In [ ]:
# ======================================================================================
# Setup — run this first (you don't need to read it closely).
# It imports a few libraries, sets a shared plot style, and defines the helper functions used
# throughout the notebook (load_data, the plot helpers, the metric helpers, ...).  These are the
# SAME helpers in every notebook, so your results line up with your teammates'.  If a later cell
# says a helper is "not defined", you probably skipped this cell — run it, then carry on.
# ======================================================================================

import os
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

# ----------------------------------------------------------------------------------------
# Paths — resolved so the notebooks work whether they're run from the materials folder or
# from solutions/ .  Outputs and figures always land in the materials root.
# ----------------------------------------------------------------------------------------
def _materials_root():
    """Folder that contains data/ ; searched upward from the working directory."""
    here = os.path.abspath(os.getcwd())
    for d in [here, os.path.dirname(here), os.path.dirname(os.path.dirname(here))]:
        if os.path.isdir(os.path.join(d, "data")):
            return d
    return here

ROOT = _materials_root()
DATA_DIR = os.path.join(ROOT, "data")
OUTPUTS_DIR = os.path.join(ROOT, "outputs")
FIGURES_DIR = os.path.join(ROOT, "figures")
MODELS_DIR = os.path.join(ROOT, "models")     # fitted models saved to disk (experiment tracking)
for _d in (DATA_DIR, OUTPUTS_DIR, FIGURES_DIR, MODELS_DIR):
    os.makedirs(_d, exist_ok=True)

# Module-level cache of the reconstruction basis (set by load_data; used by reconstruct_spectrum).
_RECON = {"wavelength": None, "basis": None, "is_mock": True, "Xerr": None}
_RAW_CACHE = {}   # path -> full arrays, so repeated load_data() calls don't re-read the file

# Default working-set size.  The real catalog is ~263k stars; we train on a fast deterministic
# subsample by default so in-class fits stay quick AND every notebook sees the SAME stars (which
# keeps the NB2->NB3->NB4 saved-prediction hand-off aligned).  Pass n=None to use the WHOLE
# high-quality catalog (the "does more data help?" / label-transfer-at-scale exercise).
DEFAULT_N = 30000

# Canonical label columns.  The modelling track predicts [Fe/H]; Teff & logg are explored and
# used as the axes for residual / coverage maps.  alpha/age/dist are extra "explore" labels.
LABEL_COLS = ["Teff", "logg", "FeH", "alpha", "age", "dist"]
PRETTY = {"Teff": r"$T_\mathrm{eff}$ [K]", "logg": r"$\log g$ [dex]",
          "FeH": r"[Fe/H] [dex]", "alpha": r"[$\alpha$/M] [dex]",
          "age": "age [Gyr]", "dist": "distance [pc]"}


# ========================================================================================
# Plot style
# ========================================================================================
def set_plot_style():
    """Consistent, readable matplotlib defaults.  Called once in setup."""
    plt.rcParams.update({
        "figure.figsize": (6.4, 4.4), "figure.dpi": 110, "savefig.dpi": 150,
        "savefig.bbox": "tight", "font.size": 12, "axes.titlesize": 13,
        "axes.labelsize": 12, "axes.grid": True, "grid.alpha": 0.25,
        "axes.axisbelow": True, "axes.spines.top": False, "axes.spines.right": False,
        "legend.frameon": False, "legend.fontsize": 11, "lines.linewidth": 2.0,
        "scatter.edgecolors": "none", "image.cmap": "viridis",
    })

# A small colour-blind-friendly palette used across the materials for named series.
COLORS = {"linear": "#888888", "knn": "#0072B2", "rf": "#009E73",
          "nn": "#D55E00", "native": "#0072B2", "conformal": "#D55E00",
          "truth": "#222222", "accent": "#CC79A7"}


# ========================================================================================
# Data loading  (the mock generator lives here so the shipped npz and the on-the-fly
# fallback are guaranteed identical)
# ========================================================================================
_C2_NM = 1.438777e7  # second radiation constant hc/k in nm*K


def _build_basis(wave_min=336.0, wave_max=1020.0, n_wave=343, n_bp=55, n_rp=55):
    """Smooth windowed-cosine modes per band, orthonormalised.  Low modes are broad
    (continuum), high modes wiggly (fine structure).  reconstruct = B @ coeff."""
    wave = np.linspace(wave_min, wave_max, n_wave)
    cols = []
    def band(lo, hi, n_modes):
        x = (wave - lo) / (hi - lo)
        win = np.where((x >= 0) & (x <= 1), np.sin(np.clip(x, 0, 1) * np.pi) ** 0.5, 0.0)
        for j in range(n_modes):
            cols.append(np.cos(j * np.pi * np.clip(x, 0, 1)) * win)
    band(wave_min, 690.0, n_bp)
    band(630.0, wave_max, n_rp)
    Q, _ = np.linalg.qr(np.column_stack(cols))
    return wave, Q[:, :n_bp + n_rp]


def _synthesize_mock(n=50000, seed=42):
    """
    # MOCK DATA — swap for the real Zenodo loader.
    Physically-motivated synthetic stand-in for the Laroche & Speagle APOGEE x Gaia-XP
    cross-match.  Returns the same dict that the shipped npz stores.  See the build spec
    and _build/make_mock_dataset.py for the full rationale.  In one sentence: a Planck
    continuum sets Teff; Balmer lines reinforce Teff; gravity-sensitive molecular bands
    carry log g; SHALLOW metal lines (suppressed at high Teff) carry the hard, heteroscedastic
    [Fe/H]; reddening + hidden factors add a realistic irreducible (aleatoric) floor.
    """
    rng = np.random.default_rng(seed)
    wave, B = _build_basis()
    W = wave.size

    # --- labels with Kiel-diagram structure: dwarfs + giants + red clump + metal-poor halo ---
    is_giant = rng.random(n) < 0.45
    Teff = np.empty(n); logg = np.empty(n); FeH = np.empty(n)
    nd, ng = np.count_nonzero(~is_giant), np.count_nonzero(is_giant)
    Td = 4000 + 3100 * rng.beta(1.5, 2.5, nd)
    Teff[~is_giant] = Td
    logg[~is_giant] = 4.62 - 0.00013 * (Td - 4500) + rng.normal(0, 0.10, nd)
    FeH[~is_giant] = rng.normal(-0.05, 0.25, nd)
    Tg = 3800 + 1700 * rng.beta(2.0, 2.3, ng)
    Teff[is_giant] = Tg
    logg[is_giant] = 1.7 + 0.00095 * (Tg - 3800) + rng.normal(0, 0.33, ng)
    FeH[is_giant] = rng.normal(-0.28, 0.34, ng)
    clump = is_giant & (rng.random(n) < 0.33); nc = np.count_nonzero(clump)
    Teff[clump] = rng.normal(4800, 140, nc); logg[clump] = rng.normal(2.45, 0.11, nc)
    FeH[clump] = rng.normal(-0.10, 0.20, nc)
    halo = rng.random(n) < 0.06; nh = np.count_nonzero(halo)
    Teff[halo] = rng.normal(4900, 450, nh)
    logg[halo] = np.clip(rng.normal(2.2, 0.7, nh), 0.5, 3.6)
    FeH[halo] = rng.normal(-1.45, 0.45, nh)
    is_giant = is_giant | halo
    Teff = np.clip(Teff, 3500, 7600); logg = np.clip(logg, 0.3, 5.0); FeH = np.clip(FeH, -2.4, 0.55)
    alpha = np.clip(0.13 - 0.24 * FeH + rng.normal(0, 0.035, n), -0.05, 0.5)
    age = np.clip(2.0 + 6.5 * is_giant - 3.5 * FeH + rng.normal(0, 1.8, n), 0.3, 13.5)
    dist = np.exp(rng.normal(6.7, 0.9, n)) * (1.0 + 1.5 * is_giant)

    # --- forward model: labels -> flux ---
    def gauss(c, w): return np.exp(-0.5 * ((wave - c) / w) ** 2)
    x = _C2_NM / (wave[None, :] * Teff[:, None])
    flux = wave[None, :] ** -5 / np.expm1(np.clip(x, 1e-6, 700))
    flux = flux / flux.mean(axis=1, keepdims=True)
    jump = 1.0 / (1.0 + np.exp((wave - 382.0) / 6.0))
    flux *= 1.0 - (0.18 * np.clip((Teff - 5200) / 2500, 0, 1.4) * (1 + 0.15 * (4.5 - logg)))[:, None] * jump[None, :]
    balmer = sum(a * gauss(c, 9.0) for c, a in [(434., .9), (486.1, 1.1), (656.3, 1.)])
    bstr = np.clip(np.exp((Teff - 5200) / 1500), 0.15, 6.0) * (1 + 0.12 * (4.5 - logg))
    flux *= 1.0 - 0.012 * bstr[:, None] * balmer[None, :]
    g_dwarf = np.clip(logg - 3.0, 0, 2.0)[:, None]; g_giant = np.clip(3.6 - logg, 0, 3.2)[:, None]
    coolf = np.clip((5500 - Teff) / 2000.0, 0.2, 1.4)[:, None]
    grav = np.zeros((n, W))
    for c, a in [(488., 1.), (521., .9), (692., .8)]: grav += a * g_dwarf * gauss(c, 15.)[None, :]
    for c, a in [(421., .9), (793., .8), (883., 1.)]: grav += a * g_giant * gauss(c, 17.)[None, :]
    flux *= 1.0 - 0.038 * coolf * grav
    metal = sum(a * gauss(c, 11.0) for c, a in
                [(422.7, 1.), (460., .7), (517.3, 1.2), (527., .9), (589.3, 1.1),
                 (670.8, .6), (770., .6), (850., .8), (920., .5)])
    # Line depth scales with metal abundance (metal-poor stars have WEAK lines -> little [Fe/H]
    # information -> the dominant, physically-correct source of heteroscedastic [Fe/H] error).
    # Temperature only mildly suppresses lines; gravity gives luminous giants slightly weaker,
    # more variable lines (they are also the sparsest, most-extrapolated regime).
    mstr = 10.0 ** (0.55 * FeH)
    tsupp = np.clip(np.exp(-(Teff - 4500) / 4800.0), 0.6, 1.25)
    line_amp = (mstr * tsupp * np.clip(0.7 + 0.22 * logg, 0.7, 1.4))[:, None]
    flux *= 1.0 - 0.045 * line_amp * metal[None, :]
    ebv = np.abs(rng.normal(0, 0.11, n))[:, None]
    ext = (550.0 / wave) ** 1.0
    flux *= 10.0 ** (-0.4 * 1.6 * ebv * (ext / ext.mean())[None, :])
    P = np.column_stack([np.sin(f * np.pi * (wave - wave[0]) / (wave[-1] - wave[0])) for f in (1.5, 3.5, 6.5)])
    flux *= 1.0 + 0.004 * (rng.normal(0, 1, (n, 3)) @ P.T)
    gflux = np.exp(rng.normal(0.0, 0.8, n))
    snr = np.clip(58 * gflux ** 0.30 * np.exp(rng.normal(0, 0.30, n)), 20, 400)
    obs = gflux[:, None] * np.clip(flux, 1e-3, None)
    noise_std = obs.mean(axis=1, keepdims=True) / snr[:, None]        # photon-noise std (per star)
    obs = obs + rng.normal(0, 1, obs.shape) * noise_std
    X = (obs @ B).astype(np.float32)
    # per-coefficient measurement error on the RAW coefficients (the orthonormal basis preserves the
    # per-component noise std) — used by the NB3 input-Monte-Carlo stretch.
    Xerr = np.repeat(noise_std, 110, axis=1).astype(np.float32)
    return dict(X=X, Xerr=Xerr, Teff=Teff, logg=logg, FeH=FeH, alpha=alpha, age=age, dist=dist,
                snr=snr, gflux=gflux, source_id=(1_000_000_000 + np.arange(n)),
                wavelength=wave, basis=B.astype(np.float32), is_mock=True)


def load_data(path=None, n=DEFAULT_N, seed=0, verbose=True):
    """
    Load the dataset in one line:  X, y, meta = load_data()

    Parameters
    ----------
    n    : working-set size.  Returns a deterministic random subsample of `n` stars (default
           DEFAULT_N=30000) so fits are fast and every notebook sees the same stars.  Pass
           `n=None` to use the ENTIRE high-quality catalog (the scale-up exercise) — slower.
    seed : seed for the subsample (keep it fixed so the NB2->NB3->NB4 hand-off stays aligned).

    Returns
    -------
    X    : float array, shape (n, 110) — the Gaia XP coefficients (55 BP + 55 RP).
    y    : DataFrame with columns Teff, logg, FeH (+ alpha, dist) — the labels.
    meta : DataFrame with source_id, snr, gflux — per-star bookkeeping (gflux is the
           brightness scale used by normalize_by_g).
    """
    if path is None:
        for cand in ("xp_apogee_real.npz", "xp_apogee_mock.npz"):
            p = os.path.join(DATA_DIR, cand)
            if os.path.exists(p):
                path = p
                break
    key = path if path is not None else "__synth__"
    if key in _RAW_CACHE:
        data = _RAW_CACHE[key]
    elif path is not None and os.path.exists(path):
        d = np.load(path, allow_pickle=True)
        data = {k: d[k] for k in d.files}
        _RAW_CACHE[key] = data
    else:
        if verbose:
            print("No data file found — synthesising the mock dataset on the fly.")
        data = _synthesize_mock()
        _RAW_CACHE[key] = data
    is_mock = bool(data.get("is_mock", True))
    src = os.path.basename(path) if path else "synthesised mock"

    _RECON["wavelength"] = np.asarray(data["wavelength"], float)
    _RECON["basis"] = np.asarray(data["basis"], float)
    _RECON["is_mock"] = is_mock

    X_all = np.asarray(data["X"], dtype=float)
    N = X_all.shape[0]
    if n is not None and n < N:
        sel = np.sort(np.random.default_rng(seed).permutation(N)[:n])
    else:
        sel = np.arange(N)
    X = X_all[sel]
    _RECON["Xerr"] = np.asarray(data["Xerr"], float)[sel] if "Xerr" in data else None  # for input-MC (NB3)
    y = pd.DataFrame({c: np.asarray(data[c], float)[sel] for c in LABEL_COLS if c in data})
    meta = pd.DataFrame({k: np.asarray(data[k]).ravel()[sel] for k in ("source_id", "snr", "gflux") if k in data})
    if verbose:
        tag = "MOCK (synthetic)" if is_mock else "REAL (Gaia XP x APOGEE)"
        extra = f" (subsampled from {N:,}; pass n=None for all)" if len(sel) < N else ""
        print(f"Loaded {src}: {tag}  —  {X.shape[0]:,} stars{extra}, {X.shape[1]} coefficients.")
    return X, y, meta


# ========================================================================================
# Spectrum reconstruction
# ========================================================================================
def load_input_errors():
    """Per-coefficient measurement errors for the stars from the LAST `load_data()` call (same rows,
    same order, same (n, 110) shape as the `X` you just loaded) — on the RAW coefficient scale.
    For the NB3 input-Monte-Carlo stretch:  X_perturbed = X + rng.normal(0, load_input_errors()).
    Returns None if the dataset has no stored errors (call `load_data()` first)."""
    if _RECON["Xerr"] is None:
        load_data(verbose=False)
    return _RECON["Xerr"]


def reconstruct_spectrum(coeffs):
    """
    Turn a star's 110 XP coefficients into a sampled spectrum: returns (wavelength_nm, flux).
    Accepts a single (110,) vector or an (M, 110) batch.

    On the shipped data this uses the stored smooth basis (B @ coeff) — no extra packages.

    # REAL DATA: with the genuine Gaia archive continuous representation you would instead
    # reconstruct with GaiaXPy, e.g.:
    #     from gaiaxpy import calibrate
    #     sampling = np.arange(336, 1021, 2.0)              # nm
    #     spectra, sampled_wl = calibrate(source_dataframe, sampling=sampling, save_file=False)
    # The stored basis is recovered from Zenodo's wavelength-space spectra at curation time
    # (see _build/prepare_real_data.py), so this helper stays identical for mock and real.
    """
    if _RECON["basis"] is None:
        load_data(verbose=False)
    B, wave = _RECON["basis"], _RECON["wavelength"]
    coeffs = np.asarray(coeffs, float)
    flux = coeffs @ B.T
    return wave, flux


# ========================================================================================
# Preprocessing recipe:  normalize by the G-band  ->  standardize (fit on train only)
# ========================================================================================
def normalize_by_g(X, g):
    """Divide each star's coefficients by its G-band brightness scale `g` (a column of meta).
    Removes brightness/distance and keeps the spectral SHAPE — what carries the labels."""
    g = np.asarray(g, float).reshape(-1, 1)
    return np.asarray(X, float) / g


def standardize(X_train, X_other):
    """Z-score the coefficients using statistics from the TRAINING set only, then apply the
    same shift/scale to another split.  Returns (X_train_scaled, X_other_scaled).
    (Equivalent to sklearn's StandardScaler fit on train; spelled out for clarity.)"""
    mu = X_train.mean(axis=0, keepdims=True)
    sd = X_train.std(axis=0, keepdims=True)
    sd = np.where(sd < 1e-12, 1.0, sd)
    return (X_train - mu) / sd, (X_other - mu) / sd


# ========================================================================================
# Splitting  (fixed seed so every student gets the same train / cal / test stars)
# ========================================================================================
def train_cal_test_split(X, y, seed=0, fracs=(0.6, 0.2, 0.2)):
    """Reproducible 60/20/20 split into train / calibration / test.
    Returns a dict with X_* arrays, y_* DataFrames, and idx_* index arrays.
    Calibration is held out for the uncertainty work in NB3/NB4."""
    X = np.asarray(X, float)
    y = y.reset_index(drop=True)
    n = X.shape[0]
    rng = np.random.default_rng(seed)
    idx = rng.permutation(n)
    n_tr = int(fracs[0] * n); n_ca = int(fracs[1] * n)
    i_tr, i_ca, i_te = idx[:n_tr], idx[n_tr:n_tr + n_ca], idx[n_tr + n_ca:]
    return {
        "X_train": X[i_tr], "X_cal": X[i_ca], "X_test": X[i_te],
        "y_train": y.iloc[i_tr].reset_index(drop=True),
        "y_cal": y.iloc[i_ca].reset_index(drop=True),
        "y_test": y.iloc[i_te].reset_index(drop=True),
        "idx_train": i_tr, "idx_cal": i_ca, "idx_test": i_te,
    }


# ========================================================================================
# Models — the four estimators the students compare.  Defined ONCE here so NB2, NB3 and NB4
# build byte-identical models, which is what lets predictions and intervals line up across
# notebooks and across the three students.  Each is a Pipeline(StandardScaler + estimator),
# so you feed it the G-normalized coefficients and it standardizes internally (fit on train).
# ========================================================================================
from sklearn.linear_model import LinearRegression
from sklearn.neighbors import KNeighborsRegressor
from sklearn.ensemble import RandomForestRegressor
from sklearn.neural_network import MLPRegressor
from sklearn.pipeline import make_pipeline
from sklearn.preprocessing import StandardScaler

MODEL_NAMES = {"linear": "Linear regression (baseline)", "knn": "K-nearest neighbors",
               "rf": "Random forest", "nn": "Neural network (MLP)"}

def make_model(name, **overrides):
    """Canonical Pipeline(StandardScaler + estimator) for a model name in {linear,knn,rf,nn}.
    Pass overrides to tune one knob, e.g. make_model('knn', n_neighbors=20).  The step name for
    the estimator is its lowercased class name (e.g. 'kneighborsregressor') — handy in NB3 when
    you reach inside the fitted pipeline for a model-native uncertainty."""
    if name == "linear":
        est = LinearRegression(**overrides)
    elif name == "knn":
        est = KNeighborsRegressor(**{"n_neighbors": 12, **overrides})
    elif name == "rf":
        est = RandomForestRegressor(**{"n_estimators": 150, "min_samples_leaf": 3,
                                       "random_state": 0, "n_jobs": -1, **overrides})
    elif name == "nn":
        est = MLPRegressor(**{"hidden_layer_sizes": (64, 64), "alpha": 1e-3, "max_iter": 500,
                              "early_stopping": True, "random_state": 0, **overrides})
    else:
        raise ValueError(f"unknown model '{name}' (use one of {list(MODEL_NAMES)})")
    return make_pipeline(StandardScaler(), est)


# ========================================================================================
# Metrics
# ========================================================================================
def regression_metrics(y_true, y_pred):
    """Return a dict of RMSE, MAE, bias (mean signed error), and R^2."""
    y_true = np.asarray(y_true, float); y_pred = np.asarray(y_pred, float)
    err = y_pred - y_true
    ss_res = float(np.sum(err ** 2))
    ss_tot = float(np.sum((y_true - y_true.mean()) ** 2)) or 1.0
    return {"RMSE": float(np.sqrt(np.mean(err ** 2))), "MAE": float(np.mean(np.abs(err))),
            "bias": float(np.mean(err)), "R2": float(1.0 - ss_res / ss_tot)}


# ========================================================================================
# Plots — predictions & residuals
# ========================================================================================
def plot_pred_vs_true(y_true, y_pred, title=None, ax=None, color=None, label=None, units=""):
    """Scatter of predicted vs true with the 1:1 line and an RMSE annotation."""
    y_true = np.asarray(y_true, float); y_pred = np.asarray(y_pred, float)
    if ax is None:
        _, ax = plt.subplots()
    lo = min(y_true.min(), y_pred.min()); hi = max(y_true.max(), y_pred.max())
    pad = 0.04 * (hi - lo + 1e-9)
    ax.plot([lo - pad, hi + pad], [lo - pad, hi + pad], "--", color="0.4", lw=1.3, zorder=1)
    ax.scatter(y_true, y_pred, s=8, alpha=0.35, color=color or COLORS["rf"], label=label, zorder=2)
    m = regression_metrics(y_true, y_pred)
    ax.text(0.04, 0.96, f"RMSE = {m['RMSE']:.3g}{units}\nbias = {m['bias']:+.2g}{units}",
            transform=ax.transAxes, va="top", ha="left", fontsize=10,
            bbox=dict(boxstyle="round", fc="white", ec="0.8", alpha=0.85))
    ax.set_xlabel(f"true{(' ' + units) if units else ''}")
    ax.set_ylabel(f"predicted{(' ' + units) if units else ''}")
    ax.set_xlim(lo - pad, hi + pad); ax.set_ylim(lo - pad, hi + pad)
    ax.set_aspect("equal", "box")
    if title:
        ax.set_title(title)
    if label:
        ax.legend(loc="lower right")
    return ax


def plot_residuals(y_true, y_pred, feature=None, feature_name="value", ax=None, color=None):
    """Residuals (pred - true) vs the truth, or vs an external `feature` (e.g. Teff).
    A flat band around zero means well-behaved errors; trends/fans flag structure."""
    y_true = np.asarray(y_true, float); y_pred = np.asarray(y_pred, float)
    resid = y_pred - y_true
    xx = np.asarray(feature, float) if feature is not None else y_true
    xlabel = feature_name if feature is not None else "true value"
    if ax is None:
        _, ax = plt.subplots()
    ax.axhline(0.0, color="0.4", ls="--", lw=1.3, zorder=1)
    ax.scatter(xx, resid, s=8, alpha=0.35, color=color or COLORS["rf"], zorder=2)
    ax.set_xlabel(xlabel); ax.set_ylabel("residual (pred − true)")
    return ax


# ========================================================================================
# Uncertainty — coverage, reliability, comparison
# ========================================================================================
def empirical_coverage(y_true, lower, upper):
    """Fraction of true values that fall inside [lower, upper]."""
    y_true = np.asarray(y_true, float)
    return float(np.mean((y_true >= np.asarray(lower, float)) & (y_true <= np.asarray(upper, float))))


def plot_reliability(y_true, lower, upper, target=None, feature=None, feature_name="bin",
                     n_bins=8, ax=None, color=None, label=None):
    """
    'Are the error bars honest, everywhere?'  Bins the points (by an external `feature` such
    as Teff if given, else by the interval centre) and plots empirical coverage per bin against
    the `target` line.  Bars sitting below the target reveal where intervals under-cover.
    """
    y_true = np.asarray(y_true, float)
    lower = np.asarray(lower, float); upper = np.asarray(upper, float)
    inside = (y_true >= lower) & (y_true <= upper)
    xx = np.asarray(feature, float) if feature is not None else 0.5 * (lower + upper)
    if ax is None:
        _, ax = plt.subplots()
    edges = np.quantile(xx, np.linspace(0, 1, n_bins + 1))
    edges[-1] += 1e-9
    centers, cov = [], []
    for i in range(n_bins):
        m = (xx >= edges[i]) & (xx < edges[i + 1])
        if m.sum() >= 5:
            centers.append(xx[m].mean()); cov.append(inside[m].mean())
    ax.plot(centers, cov, "o-", color=color or COLORS["native"], label=label)
    if target is not None:
        ax.axhline(target, color="0.4", ls="--", lw=1.3, label=f"target = {target:.0%}")
    ax.set_xlabel(feature_name if feature is not None else "interval centre")
    ax.set_ylabel("empirical coverage")
    ax.set_ylim(0, 1.02)
    if target is not None or label:
        ax.legend(loc="lower center")
    return ax


def compare_intervals(y_true, intervals, target=None, ax=None):
    """
    Compare several named interval sets.  `intervals` maps name -> (lower, upper).
    Draws two bars per method — coverage and median width — and returns a tidy summary table.
    """
    y_true = np.asarray(y_true, float)
    rows = []
    for name, (lo, hi) in intervals.items():
        rows.append({"method": name, "coverage": empirical_coverage(y_true, lo, hi),
                     "median_width": float(np.median(np.asarray(hi, float) - np.asarray(lo, float)))})
    tbl = pd.DataFrame(rows).set_index("method")
    if ax is None:
        _, ax = plt.subplots(1, 2, figsize=(9.5, 4.0))
    names = list(tbl.index)
    cols = [COLORS.get(n.split()[0].lower(), COLORS["accent"]) for n in names]
    ax[0].bar(names, tbl["coverage"], color=cols)
    if target is not None:
        ax[0].axhline(target, color="0.4", ls="--", lw=1.3, label=f"target = {target:.0%}")
        ax[0].legend()
    ax[0].set_ylabel("empirical coverage"); ax[0].set_ylim(0, 1.02); ax[0].set_title("Coverage")
    ax[1].bar(names, tbl["median_width"], color=cols)
    ax[1].set_ylabel("median interval width"); ax[1].set_title("Sharpness (narrower is better)")
    for a in ax:
        a.tick_params(axis="x", rotation=20)
    return tbl


# ========================================================================================
# Small convenience used in worked examples
# ========================================================================================
def savefig(name, fig=None):
    """Save a figure into figures/ for reuse on the poster.  Returns the path."""
    path = os.path.join(FIGURES_DIR, name)
    (fig or plt.gcf()).savefig(path)
    return path


def save_outputs(filename, **arrays):
    """Save named arrays into outputs/<filename> (the shared NB2->NB3->NB4 contract)."""
    path = os.path.join(OUTPUTS_DIR, filename)
    np.savez(path, **arrays)
    return path


def load_outputs(filename):
    """Load an outputs/ npz written by an earlier notebook; returns a dict of arrays."""
    d = np.load(os.path.join(OUTPUTS_DIR, filename), allow_pickle=True)
    return {k: d[k] for k in d.files}


# ----- Experiment tracking: save fitted models + log runs (so you can reproduce & compare) -----
def save_model(model, name):
    """Save a fitted model/pipeline to models/<name>.joblib. Returns the path.
    The point: a tuned model is an experiment result — persist it so NB3 can load the exact model
    you tuned (no silent refit), and so you can reproduce a run weeks later."""
    import joblib
    path = os.path.join(MODELS_DIR, f"{name}.joblib")
    joblib.dump(model, path)
    return path


def load_model(name):
    """Load a model previously saved with save_model(...)."""
    import joblib
    return joblib.load(os.path.join(MODELS_DIR, f"{name}.joblib"))


def log_experiment(name, params, metrics, filename=None):
    """Append one experiment run (its hyper-parameters + its metrics) to a CSV and return the full
    log as a DataFrame. This is the bare-bones version of what tools like MLflow / Weights & Biases
    do: a durable, comparable record of 'what did I try, and how did it score?'.
      params  : dict of hyper-parameters, e.g. {'n_neighbors': 12, 'weights': 'distance'}
      metrics : dict of scores, e.g. {'cv_rmse': 0.21, 'test_rmse': 0.20}
    """
    path = os.path.join(OUTPUTS_DIR, filename or f"{name}_experiments.csv")
    row = {"run": 1, **{f"param_{k}": v for k, v in params.items()}, **metrics}
    if os.path.exists(path):
        prev = pd.read_csv(path)
        row["run"] = int(prev["run"].max()) + 1 if "run" in prev else len(prev) + 1
        # Rebuild from records rather than pd.concat: a run that logs different params/metrics than
        # an earlier one would otherwise make pandas align mismatched columns into all-NA entries,
        # which raises a (noisy, harmless) FutureWarning.  Building one DataFrame from a list of
        # dicts fills the gaps with NaN silently and preserves column order.
        log = pd.DataFrame(prev.to_dict("records") + [row])
    else:
        log = pd.DataFrame([row])
    log.to_csv(path, index=False)
    return log

set_plot_style()
print('Setup complete — helpers ready.')


## Welcome to your tuning lab — you own KNN

Yesterday the whole team built the *same* linear baseline in `02_pipeline_common`. Today the three
of you **diverge**: each of you takes one model and tunes it deeply. **You own K-nearest neighbors
(KNN)** — your teammates have the random forest and the neural net.

This notebook is a **lab, not a demo**. You'll *read the docstring*, *choose the knobs*, and
*implement the search yourself*, working up through the three tools real practitioners use:

1. a **by-hand sweep** of the main knob, with a **bias–variance plot** you read off;
2. **`GridSearchCV`** — exhaustive search, perfect for a handful of categorical settings;
3. **`RandomizedSearchCV`** — when the grid gets too big to enumerate.

Along the way you'll **log every run** you try to a CSV (the bare-bones version of what tools like
MLflow or Weights & Biases do), and at the end **save your single best model** to disk so Notebook 3
can load the *exact* model you tuned — no silent refit.

> The grey **Setup** cell below has all the helpers (`load_data`, `make_model`, the plot and metric
> helpers, `log_experiment`, `save_model`, ...). **Run it, don't read it.** Everything here runs
> top-to-bottom as shipped; the "your turn" cells already have a sensible default in place — your job
> is to *change* a value and look again.


In [ ]:
# The tools this notebook uses (the Setup cell above already loaded the data/plot/log helpers).
import warnings
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from sklearn.neighbors import KNeighborsRegressor
from sklearn.model_selection import GridSearchCV, RandomizedSearchCV, cross_val_score
from scipy.stats import randint

print("Imports ready.")

## 0 · Set up your data, and recall the bar to beat

Same recipe as every notebook in this project, so your stars line up with your teammates': **load →
normalize-by-G → split with `seed=0`**. The fixed seed means you, your teammates, and the solutions
all train on the *identical* train/calibration/test stars.

We also set the per-student switch **`MODEL = "knn"`** once, here. Everything below reads `MODEL`, so
the same lab code would tune any model — but this is *your* notebook, and your model is KNN.


In [ ]:
# Load -> normalize-by-G -> split (seed=0 -> the SAME stars for the whole team).
X, y, meta = load_data()
Xn = normalize_by_g(X, meta["gflux"])          # physics step: divide out brightness, keep spectral SHAPE
sp = train_cal_test_split(Xn, y, seed=0)
print("train / cal / test sizes:",
      sp["X_train"].shape[0], sp["X_cal"].shape[0], sp["X_test"].shape[0])

target = "FeH"                                  # our prediction target is metallicity, [Fe/H]
y_train = sp["y_train"][target].to_numpy()
y_cal   = sp["y_cal"][target].to_numpy()
y_test  = sp["y_test"][target].to_numpy()

MODEL = "knn"                                   # you own KNN (A:"knn"  B:"rf"  C:"nn")
print(f"You are tuning MODEL = {MODEL!r} to predict {target}.")

# a small slice for FAST tuning — search here, fit the final model on the full split
_ti = np.random.default_rng(0).choice(sp["X_train"].shape[0], size=min(5000, sp["X_train"].shape[0]), replace=False)
X_tune, y_tune = sp["X_train"][_ti], sp["y_train"]["FeH"].to_numpy()[_ti]
print(f"tuning slice: {X_tune.shape[0]} of {sp['X_train'].shape[0]} train stars (searches run here for speed)")

In [ ]:
# The bar to beat: the linear baseline from 02_pipeline_common. Load its saved test predictions if
# they're there; otherwise refit it in two lines (so this notebook stands alone).
try:
    d = load_outputs("linear_preds.npz")
    baseline_rmse = regression_metrics(d["y_test_true"], d["y_test_pred"])["RMSE"]
    print(f"loaded linear baseline from outputs/  ->  test RMSE = {baseline_rmse:.3f} dex")
except FileNotFoundError:
    lin = make_model("linear").fit(sp["X_train"], y_train)
    baseline_rmse = regression_metrics(y_test, lin.predict(sp["X_test"]))["RMSE"]
    print(f"(no saved baseline found — refit on the fly)  ->  test RMSE = {baseline_rmse:.3f} dex")

print(f"\nGoal for the day: get your tuned KNN below ~{baseline_rmse:.2f} dex, and learn WHERE it wins.")

*Caption: the linear baseline lands around **0.2 dex** RMSE on `[Fe/H]`. That's the number your tuned KNN has to clear — and, more interestingly, you'll find out *where* it does and doesn't.*

## 1 · Meet your estimator — K-nearest neighbors

KNN is the most intuitive model in the project. To predict a new star's `[Fe/H]`, it finds the **k
most similar stars** in the training set (nearest in the 110-dimensional space of standardized XP
coefficients) and **averages their `[Fe/H]`**. "You are like your closest neighbors." There's no
"training" in the usual sense — fitting just *stores* the training stars; the work happens at predict
time, when it searches for neighbors.

First, the professional habit: **read the docstring.** The next cell prints scikit-learn's own
documentation for `KNeighborsRegressor`. Skim it for the parameter names — those are the knobs you'll
turn. (In a live notebook you can also type `KNeighborsRegressor?` to pop up the same help.)


In [ ]:
# Read the docs straight from the source. (help() prints the full docstring; we trim it so the
# output stays short. Drop the [:HERE] slice — or run  KNeighborsRegressor?  — to see all of it.)
import io, contextlib
buf = io.StringIO()
with contextlib.redirect_stdout(buf):
    help(KNeighborsRegressor)
doc = buf.getvalue()
print(doc[:1500], "\n... (truncated — see the Parameters section above for every knob)")

### The knobs that matter (your tuning guide)

Out of the full docstring, four knobs are worth your time. They come in the three *kinds* every model
has — **ordinal** (a number with an order), **categorical** (a choice from a set), and "structural"
(here, the distance *geometry*):

| Knob | Kind | What it does | What you'll do with it |
|---|---|---|---|
| `n_neighbors` (**k**) | ordinal | how many neighbors to average | **§2** — sweep it by hand, plot bias–variance |
| `weights` | categorical | `"uniform"` (all k count equally) vs `"distance"` (closer stars count more) | **§3** — compare |
| `p` (with `metric="minkowski"`) | categorical/structural | `p=2` = Euclidean (straight-line) distance, `p=1` = Manhattan (city-block) | **§3** — compare |
| `algorithm` | (speed only) | `"ball_tree"`/`"kd_tree"`/`"brute"` — how neighbors are *found* | leave at `"auto"`; see the note on dimensionality |

**The big idea for k:** small **k** = a *flexible* model that can chase noise (overfit); large **k** =
a *smooth* model that washes out real structure (underfit). You're hunting the sweet spot in between,
judged on held-out data — never on the training set. That's the whole of §2.

> **Curse of dimensionality (worth knowing).** KNN judges similarity by *distance*, and in **110
> dimensions** distances get less discriminating — points drift toward being roughly equidistant, so
> "nearest" means a little less than it would in 2-D. This is exactly *why we standardize first* (so no
> one coefficient dominates the distance), and why `algorithm` is only a *speed* lever here: in 110-D the
> fancy tree structures fall back to effectively brute-force search anyway, so it won't change your
> scores — only the runtime. (An "Explore" idea at the end: does reducing to a few PCA components help
> KNN?)

*Reference:* [KNeighborsRegressor — API](https://scikit-learn.org/stable/modules/generated/sklearn.neighbors.KNeighborsRegressor.html).


In [ ]:
# Fit KNN ONCE with the defaults (make_model("knn") uses n_neighbors=12) to get a starting score,
# then LOG it. Logging every run is the habit of the day: a durable record of "what did I try, and
# how did it score?" -- the bare-bones version of MLflow / Weights & Biases.

# Start today's log fresh so the run numbering is clean (log_experiment APPENDS, so an old CSV would
# keep growing across re-runs). Comment this out if you'd rather keep accumulating across sessions.
_log_path = os.path.join(OUTPUTS_DIR, f"{MODEL}_experiments.csv")
if os.path.exists(_log_path):
    os.remove(_log_path)

start_pipe = make_model(MODEL)                       # Pipeline(StandardScaler + KNeighborsRegressor(k=12))
start_pipe.fit(sp["X_train"], y_train)
start_metrics = regression_metrics(y_test, start_pipe.predict(sp["X_test"]))
print("KNN with default settings (k=12, weights=uniform, p=2):")
for k_, v_ in start_metrics.items():
    print(f"  {k_:>5}: {v_: .4f}")

# Record this run. params = the knobs; metrics = the scores. This APPENDS a row to
# outputs/knn_experiments.csv and returns the whole log so far.
log = log_experiment(MODEL,
                     {"n_neighbors": 12, "weights": "uniform", "p": 2, "source": "defaults"},
                     {"test_rmse": start_metrics["RMSE"], "test_r2": start_metrics["R2"]})
print(f"\nlogged run #{int(log['run'].max())} to outputs/{MODEL}_experiments.csv")

*Caption: out of the box KNN already gives a test RMSE you can compare to the baseline's ~0.2 dex. Don't over-read one number — the rest of the lab is about choosing the knobs honestly and seeing where the model is trustworthy.*

## 2 · Sweep the main knob by hand — and *see* the bias–variance tradeoff

Time to tune `n_neighbors` (k) yourself. The honest way is **cross-validation on the training data**:
split the training set into a few **folds**, rotate which fold is scored, average the result. The
**test set stays untouched** until the very end. We use `cross_val_score` with `cv=3` (three folds —
fast and plenty for a clear curve).

For each k you'll record **two** numbers:

- **train RMSE** — the model scored on the very stars it stored (optimistic by design), and
- **CV RMSE** — the honest held-out estimate.

Plot both against k and a story appears. At **small k** the train RMSE plunges toward zero (with
`k=1` the model just returns each star's own label — *perfect* memorization) while the CV RMSE is
high: that gap **is** overfitting. At **large k** both rise together as the model over-smooths: that's
underfitting. The CV curve dips lowest in between — your sweet spot.

> One sign convention: scikit-learn *maximizes* its score, so RMSE comes back **negated**
> (`neg_root_mean_squared_error`). Flip the sign to read it as an RMSE.


In [ ]:
# <- your turn: sweep k and watch train vs CV RMSE. The grid below runs as-is; widen it (try adding
#    1, 2, 80, 120) and re-run to see the ends of the curve more clearly.
k_values = [1, 3, 5, 8, 12, 20, 35, 60]          # <- your turn: edit this list

train_rmses, cv_rmses = [], []
for k in k_values:
    pipe_k = make_model(MODEL, n_neighbors=k)
    pipe_k.fit(X_tune, y_tune)
    tr = regression_metrics(y_tune, pipe_k.predict(X_tune))["RMSE"]   # optimistic (same stars)
    cv = -cross_val_score(make_model(MODEL, n_neighbors=k), X_tune, y_tune,
                          cv=3, scoring="neg_root_mean_squared_error", n_jobs=-1).mean()
    train_rmses.append(tr); cv_rmses.append(cv)
    log_experiment(MODEL, {"n_neighbors": k, "weights": "uniform", "p": 2, "source": "k_sweep"},
                   {"train_rmse": tr, "cv_rmse": cv})
    print(f"  k={k:3d}   train RMSE={tr:.4f}   CV RMSE={cv:.4f}")

best_k = k_values[int(np.argmin(cv_rmses))]
print(f"\nlowest CV RMSE at k = {best_k}")

In [ ]:
# Plot the bias-variance curve: train RMSE vs CV RMSE across k. (Smaller k is to the RIGHT here so
# "more flexible" reads left-to-right like a flexibility dial; we flip the x-axis to make that natural.)
fig, ax = plt.subplots(figsize=(6.6, 4.6))
ax.plot(k_values, train_rmses, "o-", color="0.6", label="train RMSE (optimistic)")
ax.plot(k_values, cv_rmses, "o-", color=COLORS["knn"], label="cross-val RMSE (honest)")
ax.axhline(baseline_rmse, color="0.4", ls="--", lw=1.3, label=f"linear baseline (~{baseline_rmse:.2f})")
ax.axvline(best_k, color=COLORS["accent"], ls=":", lw=1.5, label=f"best k = {best_k}")
ax.invert_xaxis()                                   # small k (flexible) on the right
ax.set_xlabel("n_neighbors  (k)   —  flexible ←→ smooth")
ax.set_ylabel("RMSE  [dex]")
ax.set_title("KNN bias–variance: train vs cross-validation")
ax.legend(loc="upper center", fontsize=9)
savefig("nb2a_knn_biasvariance.png")
plt.show()

# <- your turn: which end is OVERFITTING and which is UNDERFITTING?
#    Look at the GAP between the grey (train) and blue (CV) curves at small k.

*Caption: classic bias–variance. At small **k** the train curve dives toward zero while the CV curve stays high — the big gap is **overfitting**. At large **k** both rise together — **underfitting**. The honest (CV) curve bottoms out in the middle: that's your k. Notice it already sits below the baseline line.*

## 3 · Categorical knobs — `weights` and the distance metric

`n_neighbors` is ordinal — a dial you slide. The other two knobs are **categorical** — a choice from a
small set, where a sweep-and-plot doesn't make sense; you just **try each option and compare**.

- **`weights`** — `"uniform"` (all k neighbors count equally) vs `"distance"` (closer neighbors count
  *more*, weighted by 1/distance). `"distance"` often helps a little, especially at larger k, because a
  far-away "neighbor" shouldn't get an equal vote.
- **`p`** (the Minkowski power, with the default `metric="minkowski"`) — **`p=2`** is ordinary
  **Euclidean** (straight-line) distance; **`p=1`** is **Manhattan** (city-block) distance, summing
  absolute differences. In high dimensions Manhattan is sometimes a touch more robust.

Hold k fixed (use your `best_k` from §2) and score all **2 × 2 = 4** combinations with the same 3-fold
CV. Log each, then read off the winner.


In [ ]:
# <- your turn: try the 2 x 2 grid of categorical options at your best_k, and log each.
#    Change best_k, or add a metric, and re-run to see what moves.
rows = []
for w in ["uniform", "distance"]:                 # <- your turn: both are worth trying
    for p in [2, 1]:                               # p=2 Euclidean, p=1 Manhattan
        cv = -cross_val_score(make_model(MODEL, n_neighbors=best_k, weights=w, p=p),
                              X_tune, y_tune, cv=3,
                              scoring="neg_root_mean_squared_error", n_jobs=-1).mean()
        metric_name = {2: "euclidean", 1: "manhattan"}[p]
        rows.append({"weights": w, "p": p, "metric": metric_name, "cv_rmse": cv})
        log_experiment(MODEL, {"n_neighbors": best_k, "weights": w, "p": p, "source": "categorical"},
                       {"cv_rmse": cv})

cat_table = pd.DataFrame(rows).sort_values("cv_rmse").reset_index(drop=True)
print(cat_table.to_string(index=False))
best_row = cat_table.iloc[0]
print(f"\nbest categorical combo: weights={best_row['weights']!r}, "
      f"metric={best_row['metric']} (p={int(best_row['p'])})")

*Caption: a tiny scorecard. The differences here are **small** — categorical knobs rarely move the needle as much as k does — but `weights="distance"` and/or Manhattan (`p=1`) tend to edge ahead. Note which combo lands on top for *your* run.*

## 4 · Systematic search — `GridSearchCV`, then `RandomizedSearchCV`

So far you've tuned k *and* the categorical knobs **separately**. But the best k might differ once you
also pick `weights` and `p` — the knobs interact. `GridSearchCV` handles that: hand it a grid and it
**cross-validates every combination** and refits the best one for you, all leakage-free (the scaler
inside the pipeline refits on each fold).

A subtlety: because the estimator lives inside a `Pipeline`, you address its parameters with the
**`step__param`** double-underscore naming. The step is named for its lowercased class — so it's
`kneighborsregressor__n_neighbors`, etc. (Print `make_model("knn").named_steps` to see the names.)

Keep the executed grid **modest** so this runs in a minute or two — but the prose is inviting you to
widen it once you see how it works.


In [ ]:
# <- your turn: build a small grid over k x weights x p and run GridSearchCV with cv=3.
#    Widen the lists (more k values, etc.) once it's working -- the cost grows with the PRODUCT of sizes.
step = "kneighborsregressor"                       # the estimator's step name inside the pipeline
grid = {
    f"{step}__n_neighbors": [5, 12, 20, 35],       # <- your turn: add/remove values
    f"{step}__weights":     ["uniform", "distance"],
    f"{step}__p":           [1, 2],
}
gs = GridSearchCV(make_model(MODEL), grid, cv=3,
                  scoring="neg_root_mean_squared_error", n_jobs=-1)
gs.fit(X_tune, y_tune)                              # 4*2*2 = 16 combos x 3 folds = 48 fits

print("best params:", gs.best_params_)
print(f"best CV RMSE: {-gs.best_score_:.4f} dex")

# Score the grid's chosen model on the UNTOUCHED test set, and log it.
grid_pred = gs.best_estimator_.predict(sp["X_test"])
grid_rmse = regression_metrics(y_test, grid_pred)["RMSE"]
print(f"grid-best test RMSE: {grid_rmse:.4f} dex")
_ = log_experiment(MODEL, {**{k.split("__")[1]: v for k, v in gs.best_params_.items()}, "source": "grid"},
               {"cv_rmse": -gs.best_score_, "test_rmse": grid_rmse})

*Caption: `GridSearchCV` tried every k × weights × p combination with honest cross-validation and refit the winner. Its test RMSE should be a touch below what k-alone gave you — the categorical knobs chipped in. This is the idiom to reach for whenever you have a few knobs with a handful of values each.*

### When the grid explodes — `RandomizedSearchCV`

A full grid is exhaustive, which is great until it isn't: add `min_samples`-style knobs, or let k run
over *every* value 1–60 with both metrics and both weightings, and the number of combinations blows
up. **`RandomizedSearchCV`** is the escape hatch — you give it *distributions* (or lists) to sample
from and a fixed **budget** (`n_iter`), and it tries that many random combinations. You trade
guaranteed-exhaustive for "very good, much cheaper," which is the right trade once the space is large.

Here we let k be *any* integer in **3–60** (a `randint` distribution) and still sample `weights` and
`p` — a space far too big to grid — but cap it at a handful of draws so it stays fast.


In [ ]:
# <- your turn: sample a BIGGER space cheaply. n_iter is your budget -- raise it for a better search
#    (and a longer wait). random_state makes the draw reproducible.
param_dist = {
    "kneighborsregressor__n_neighbors": randint(3, 61),     # any integer 3..60
    "kneighborsregressor__weights":     ["uniform", "distance"],
    "kneighborsregressor__p":           [1, 2],
}
rs = RandomizedSearchCV(make_model(MODEL), param_dist, n_iter=8, cv=3,
                        scoring="neg_root_mean_squared_error",
                        n_jobs=-1, random_state=0)                # <- your turn: raise n_iter
rs.fit(X_tune, y_tune)

print("best params:", rs.best_params_)
print(f"best CV RMSE: {-rs.best_score_:.4f} dex  (from {rs.n_iter} random draws)")
_ = log_experiment(MODEL, {**{k.split("__")[1]: v for k, v in rs.best_params_.items()}, "source": "random"},
               {"cv_rmse": -rs.best_score_})

*Caption: with only a handful of random draws, the randomized search lands *close* to the grid's answer — for a fraction of the combinations it would take to grid the same range exhaustively. That's the whole pitch: near-best, much cheaper, once the space is large.*

## 5 · Your tuned model — the honest test score and the error map

You've done your tuning on cross-validation. Now — **once** — you fit the best settings and score on
the **untouched test set**. (We've peeked at test RMSE a couple of times above for teaching; the
*honest* protocol is: tune on CV, report once on test. In a real project you'd resist looking until
the end.)

Then the part that matters most for the science: **a single RMSE hides *where* the model fails.** We
map the absolute `[Fe/H]` error across the **Kiel diagram** (the `Teff`–`log g` plane astronomers use —
hot on the left, luminous giants at the top), and we bin the error by `log g` and by `[Fe/H]`. **Let
the numbers tell you where it breaks** — don't assume.


In [ ]:
# <- your turn: set your tuned knobs (paste your best from the grid in section 4), fit, score on TEST.
best_pipe = make_model(MODEL, n_neighbors=12, weights="distance", p=1)   # <- your turn: YOUR best params
best_pipe.fit(sp["X_train"], y_train)

test_pred = best_pipe.predict(sp["X_test"])
tuned = regression_metrics(y_test, test_pred)
print(f"TUNED KNN on the untouched test set:")
for k_, v_ in tuned.items():
    print(f"  {k_:>5}: {v_: .4f}")
print(f"\nlinear baseline was {baseline_rmse:.4f} dex  ->  "
      f"you {'BEAT' if tuned['RMSE'] < baseline_rmse else 'did NOT beat'} it "
      f"by {baseline_rmse - tuned['RMSE']:+.4f} dex.")
_ = log_experiment(MODEL, {"n_neighbors": 12, "weights": "distance", "p": 1, "source": "FINAL"},
               {"test_rmse": tuned["RMSE"], "test_r2": tuned["R2"], "test_mae": tuned["MAE"]})

*Caption: your tuned KNN's one honest test score. It should sit a little below the ~0.2-dex baseline. The headline isn't the margin, though — it's the *shape* of the errors, which the next two cells map.*

In [ ]:
# Two diagnostics side by side, for the tuned model on the TEST stars.
fig, axes = plt.subplots(1, 2, figsize=(11, 4.4))
plot_pred_vs_true(y_test, test_pred, ax=axes[0],
                  color=COLORS["knn"], title="Tuned KNN — [Fe/H]", units=" dex")
plot_residuals(y_test, test_pred, feature=sp["y_test"]["Teff"],
               feature_name=r"$T_\mathrm{eff}$ [K]", ax=axes[1], color=COLORS["knn"])
axes[1].set_title("Residuals vs temperature")
plt.tight_layout()
savefig("nb2a_knn_pred.png")
plt.show()

*Caption: points hug the 1:1 line, but the residual band isn't a uniform stripe — its width changes across the data. A single RMSE can't see that. The error map below pins down *which stars* drive it.*

In [ ]:
# Absolute [Fe/H] error per TEST star, placed on the Teff-logg plane (the Kiel diagram).
abs_err = np.abs(test_pred - y_test)
teff = sp["y_test"]["Teff"].to_numpy()
logg = sp["y_test"]["logg"].to_numpy()
feh  = sp["y_test"]["FeH"].to_numpy()

fig, ax = plt.subplots(figsize=(6.8, 5.2))
sc = ax.scatter(teff, logg, c=abs_err, s=10, cmap="viridis",
                vmax=np.quantile(abs_err, 0.95))    # clip color so a few big errors don't wash it out
ax.invert_xaxis(); ax.invert_yaxis()                # Kiel convention: hot left, giants top
ax.set_xlabel(r"$T_\mathrm{eff}$ [K]"); ax.set_ylabel(r"$\log g$ [dex]")
ax.set_title("Tuned KNN: |[Fe/H] error| across the Kiel plane")
fig.colorbar(sc, label="|pred - true|  [dex]")
savefig("nb2a_knn_errormap.png")
plt.show()

# <- your turn: where are the errors largest? The luminous giants up top (low log g)?
#    The metal-poor stars? Try coloring by SIGNED error (test_pred - y_test) -- biased high or low?

*Caption: the error is clearly **not uniform** across the plane. Don't eyeball it alone — the next cell bins the error by `log g` and by `[Fe/H]` so you can read off exactly which stars are hard.*

In [ ]:
# Quantify the discovery: mean |error| in terciles of log g and of [Fe/H].
def binned_error(values, name, n=3):
    edges = np.quantile(values, np.linspace(0, 1, n + 1)); edges[-1] += 1e-9
    print(f"mean |[Fe/H] error| by {name}:")
    for i in range(n):
        m = (values >= edges[i]) & (values < edges[i + 1])
        print(f"  {name} in [{edges[i]:7.2f}, {edges[i+1]:7.2f}] : "
              f"{abs_err[m].mean():.3f} dex   (n={m.sum()})")

binned_error(logg, "log g")    # low log g = luminous GIANTS; high log g = compact DWARFS
binned_error(feh,  "[Fe/H]")   # low [Fe/H] = METAL-POOR stars

*Caption: read the two tables together. The error is largest for the **lowest-`log g`** stars (luminous giants) and the **lowest-`[Fe/H]`** stars (metal-poor) — the sparse, information-starved corners, where weak metal lines carry little signal. That uneven, *conditional* error is exactly the problem tomorrow's uncertainty work takes on.*

## 6 · Interpret your model with SHAP (stretch)

In Notebook 1 you read a linear model's "effects" straight off its coefficients. KNN has **no
coefficients** — it's just stored stars and a distance — so you can't peek inside the same way. This is
where a model-agnostic tool earns its keep: **SHAP** (SHapley Additive exPlanations) asks, for each
prediction, *how much did each of the 110 coefficients push it up or down* relative to a baseline,
using a fair credit-assignment rule borrowed from game theory.

The catch for KNN: there's **no fast exact explainer** (unlike random forests, which have one). SHAP
has to *probe* the model by calling `predict` on many perturbed inputs — and each KNN `predict` does a
neighbor search. So this is **slow**, and we keep it honest by capping hard: a **small background**
(~50 reference stars), only a **handful of test stars explained** (~25), and a capped number of probes
(`nsamples`). That's enough to see *which coefficients matter*; widen any of these only if you're
willing to wait.

> This is the same idea as NB1's hand-computed linear effects, generalized to a model you can't read
> coefficients off. The output is per-star, per-feature attributions whose sum (plus the baseline)
> reconstructs each prediction.


In [ ]:
import shap

# Keep it FAST: small background + few explained rows + capped probes. (KNN has no fast explainer, so
# SHAP must call predict() many times -- each call is a neighbor search. These caps keep it ~1-2 min.)
rng = np.random.default_rng(0)
background = shap.sample(sp["X_train"], 100, random_state=0)   # ~100 reference stars
n_explain  = 50                                               # explain ~50 test stars (raise to wait longer)
X_explain  = sp["X_test"][:n_explain]                         # a small sample of the test set

with warnings.catch_warnings():
    warnings.simplefilter("ignore")
    explainer = shap.KernelExplainer(best_pipe.predict, background)
    shap_values = explainer.shap_values(X_explain, nsamples=100, silent=True)   # 100 probes per star

shap_values = np.asarray(shap_values)
print("SHAP values shape (stars x coefficients):", shap_values.shape)

*Caption: one SHAP value per coefficient per explained star. They're additive: each star's attributions sum (with the background baseline) to its predicted `[Fe/H]`. This cell is the slow one — a minute or two — because every probe is a neighbor search.*

In [ ]:
# Which coefficients drive KNN's [Fe/H] predictions? Rank by MEAN ABSOLUTE SHAP (a global importance).
mean_abs_shap = np.abs(shap_values).mean(axis=0)            # average |attribution| over the explained stars
order = np.argsort(mean_abs_shap)[::-1]
top = order[:12]

fig, ax = plt.subplots(figsize=(7.2, 4.6))
ax.barh([f"coef {i}" for i in top][::-1], mean_abs_shap[top][::-1], color=COLORS["knn"])
ax.set_xlabel("mean |SHAP value|   [dex]")
ax.set_title("KNN: which XP coefficients drive [Fe/H]?  (top 12)")
plt.tight_layout()
savefig("nb2a_knn_shap.png")
plt.show()

print("top coefficients by mean|SHAP|:", top.tolist())
# <- your turn: a handful of the 110 coefficients carry most of the [Fe/H] signal. (Coefficients
#    0-54 are the BP band, 55-109 the RP band.) Does that hint at WHERE the metal lines live?

*Caption: a **few** of the 110 coefficients carry most of KNN's `[Fe/H]` signal — the rest barely move predictions. That sparsity matches the physics: metallicity lives in a handful of shallow metal lines, not the whole spectrum. (This is exactly why `[Fe/H]` is hard and why the curse of dimensionality bites — most of the 110-D distance is noise.)*

## 7 · Save your work — model, predictions, and the experiment log

Three things to persist, so tomorrow's notebooks can pick up *exactly* the model you tuned:

1. **The fitted model** → `save_model(best_pipe, "knn")` writes `models/knn.joblib`. Notebook 3 will
   `load_model("knn")` to get your tuned KNN back — **no silent refit**, so the model you quantify is
   the model you tuned.
2. **The predictions** → `save_outputs("knn_preds.npz", ...)` with the locked keys
   `y_cal_true, y_cal_pred, y_test_true, y_test_pred`. NB3/NB4 read these to build intervals. (We save
   the *calibration* predictions too because tomorrow's uncertainty work is sized on them.)
3. **The experiment log** → you've been appending to `outputs/knn_experiments.csv` all along; let's
   look at it and confirm your best run.

> Filenames and keys are a **contract** — don't rename them, or the later notebooks won't find your
> work. The model string for the neural net is `"nn"`, not `"mlp"`; yours is `"knn"`.


In [ ]:
# 1) Save the fitted tuned model so NB3 can load the EXACT model you tuned.
path = save_model(best_pipe, MODEL)
print("wrote", path)

# 2) Save predictions on BOTH the calibration and test splits (the NB2->NB3->NB4 contract).
save_outputs(
    f"{MODEL}_preds.npz",
    y_cal_true =y_cal,
    y_cal_pred =best_pipe.predict(sp["X_cal"]),     # predictions on the held-out CALIBRATION split
    y_test_true=y_test,
    y_test_pred=best_pipe.predict(sp["X_test"]),    # predictions on the TEST split
)
print(f"wrote outputs/{MODEL}_preds.npz")

# sanity check: reload and confirm the schema NB3/NB4 expect.
chk = load_outputs(f"{MODEL}_preds.npz")
print("keys:", sorted(chk.keys()), "| test preds shape:", chk["y_test_pred"].shape)

*Caption: your tuned KNN is now on disk (`models/knn.joblib`) along with its cal+test predictions (`outputs/knn_preds.npz`) in the exact schema NB3/NB4 read. The linear baseline was already saved by the common notebook.*

In [ ]:
# 3) Look at your experiment log -- every run you tried today, in one comparable table.
#    (log_experiment has been appending each run to this CSV; now we just read it back with pandas.)
exp = pd.read_csv(os.path.join(OUTPUTS_DIR, f"{MODEL}_experiments.csv"))
print(f"outputs/{MODEL}_experiments.csv  —  {len(exp)} runs logged today\n")
# Show the runs that recorded a test RMSE, best first.
if "test_rmse" in exp.columns:
    scored = exp.dropna(subset=["test_rmse"]).sort_values("test_rmse")
    print(scored.to_string(index=False))
    print(f"\nbest test RMSE on record: {scored['test_rmse'].min():.4f} dex")
else:
    print(exp.tail(10).to_string(index=False))

*Caption: this CSV is your lab notebook — a durable, comparable record of every setting you tried and how it scored. Real teams use tools like **MLflow** or **Weights & Biases** for exactly this, at scale; `log_experiment` is the same idea boiled down to one function. Skim it and confirm your saved model is the best one.*

## Wrap-up · what you tuned, and what's next

You ran a real tuning lab on **K-nearest neighbors**, end to end:

- read the docstring and mapped the **knobs that matter** (ordinal `k`; categorical `weights` and the
  distance metric `p`);
- swept **k by hand** and *read the bias–variance curve* — `k=1` memorizes (train RMSE 0, overfit),
  large `k` over-smooths (underfit), and the honest CV minimum sits in between;
- compared categorical knobs, then ran **`GridSearchCV`** (exhaustive) and **`RandomizedSearchCV`**
  (budget-capped, for big spaces);
- **logged every run** to a CSV, **scored once** on the untouched test set (beating the ~0.2-dex linear
  baseline), and mapped *where* the error concentrates — **metal-poor stars and luminous giants**, the
  sparse corners where KNN has few good neighbors;
- used **SHAP** to see which coefficients drive a model that has no coefficients to read;
- and **saved** your tuned model + predictions for tomorrow.

That conditional error map is the cliffhanger. **Notebook 3 (`03a_uncertainty_knn`)** turns your
predictions into **intervals** and asks whether they're *honest everywhere* — and you'll find the
giants and metal-poor stars under-covered. Beautifully, KNN's *own structure* hands you an uncertainty
for free: the **scatter in `[Fe/H]` among a star's k neighbors**. The same sparse corners that hurt
accuracy here are where that native uncertainty grows — which is exactly the point. **Notebook 4** then
makes the coverage *guarantee* rigorous with conformal prediction.

Bring your bias–variance plot, your error map, and a one-line "KNN vs the baseline" verdict to debrief.


## Further reading

**scikit-learn — the tools you used today**
- [KNeighborsRegressor — API reference](https://scikit-learn.org/stable/modules/generated/sklearn.neighbors.KNeighborsRegressor.html) — `n_neighbors`, `weights`, `metric`/`p`, `algorithm`.
- [Nearest Neighbors — User Guide](https://scikit-learn.org/stable/modules/neighbors.html) — how neighbor search works and when KNN shines or struggles.
- [Tuning hyper-parameters: GridSearchCV & RandomizedSearchCV](https://scikit-learn.org/stable/modules/grid_search.html) — exhaustive vs randomized search, and the `cv` / `scoring` knobs.
- [Cross-validation: evaluating estimator performance](https://scikit-learn.org/stable/modules/cross_validation.html) — why you tune on CV and report once on test.
- [Pipelines: chaining estimators](https://scikit-learn.org/stable/modules/compose.html#pipeline) — why the scaler-in-a-Pipeline makes CV leakage-free, and the `step__param` naming.
- [Underfitting vs. overfitting (worked example)](https://scikit-learn.org/stable/auto_examples/model_selection/plot_underfitting_overfitting.html) — the bias–variance picture you drew for k, in another setting.
- [The curse of dimensionality (sklearn glossary / nearest-neighbor notes)](https://scikit-learn.org/stable/modules/neighbors.html#nearest-neighbor-algorithms) — why distances get less discriminating in high dimensions.

**Interpretability**
- [SHAP documentation](https://shap.readthedocs.io/en/latest/) — model-agnostic explanations; see KernelExplainer for models (like KNN) with no fast exact explainer.
- [A Unified Approach to Interpreting Model Predictions (Lundberg & Lee 2017)](https://arxiv.org/abs/1705.07874) — the SHAP paper.

**Experiment tracking (what real teams use)**
- [MLflow Tracking](https://mlflow.org/docs/latest/tracking.html) — log params, metrics, and models at scale.
- [Weights & Biases — Experiment Tracking](https://docs.wandb.ai/guides/track) — the same idea, hosted.

**The science**
- [Laroche & Speagle 2025 — stellar labels from Gaia XP spectra (arXiv:2404.07316)](https://arxiv.org/abs/2404.07316) — the cross-match and the reason `[Fe/H]` is the hard label.
- [Gaia DR3 BP/RP spectra (ESA image-of-the-week)](https://www.cosmos.esa.int/web/gaia/iow_20220131) — what the 55+55 XP coefficients are.
